# 🛢️ Global Fuel Price Predictor

**Tugas Besar — Dasar Ilmu Data (Semester 3)**

Notebook ini berisi pipeline lengkap untuk memprediksi harga bahan bakar (petrol) global menggunakan tiga model Machine Learning:

| Model | Metode Tuning |
|-------|---------------|
| **K-Nearest Neighbours (KNN)** | GridSearchCV |
| **Support Vector Regression (SVM)** | GridSearchCV (subsampled) |
| **Random Forest** | RandomizedSearchCV |

---

## Daftar Isi
1. Setup & Instalasi
2. Upload / Mount Dataset
3. Eksplorasi Data (EDA)
4. Preprocessing
5. Model — KNN
6. Model — SVM
7. Model — Random Forest
8. Training Pipeline
9. Perbandingan Model
10. Kesimpulan

---
## 1. Setup & Instalasi

In [ ]:
# Dependencies (semua sudah pre-installed di Colab)
import os
import json
import time
import logging
import warnings
from datetime import datetime
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    train_test_split,
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR, LinearSVR

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] %(levelname)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("fuel_price_predictor")

RANDOM_STATE = 42

print("✅ Semua library berhasil di-import!")
print(f"   NumPy {np.__version__}")
print(f"   Pandas {pd.__version__}")
print(f"   Matplotlib {matplotlib.__version__}")
print(f"   Scikit-learn {__import__('sklearn').__version__}")

---
## 2. Upload / Mount Dataset

Pilih **salah satu** metode di bawah ini:

In [ ]:
# ===== OPSI A: Upload langsung dari komputer =====
from google.colab import files

print("📁 Silakan upload file 'global_fuel_prices_2020_2026.csv'")
uploaded = files.upload()

DATA_PATH = list(uploaded.keys())[0]
print(f"\n✅ File berhasil di-upload: {DATA_PATH}")

In [ ]:
# ===== OPSI B: Mount Google Drive =====
# Uncomment cell ini jika file CSV ada di Google Drive

# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/path/to/global_fuel_prices_2020_2026.csv'

In [ ]:
# Load dataset
df = pd.read_csv(DATA_PATH)
print(f"📊 Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"   Kolom: {list(df.columns)}")
df.head()

---
## 3. Eksplorasi Data (EDA)

### 3.1 Informasi Dataset

In [ ]:
print("=" * 60)
print("INFORMASI DATASET")
print("=" * 60)
df.info()
print("\n")
print("Missing Values:")
print(df.isnull().sum())
print(f"\nJumlah negara unik: {df['country'].nunique()}")
print(f"Jumlah region unik: {df['region'].nunique()}")
print(f"Rentang tanggal: {df['date'].min()} s/d {df['date'].max()}")

### 3.2 Statistik Deskriptif

In [ ]:
TARGET = "petrol_usd_liter"

numeric = df.select_dtypes(include=[np.number])
stats = numeric.describe().T.reset_index().rename(columns={"index": "feature"})
for col in stats.columns:
    if col != "feature":
        stats[col] = stats[col].round(3)
stats

### 3.3 Visualisasi — Design System

In [ ]:
# ── Design System Palette ──
PRIMARY    = "#17171c"     # near-black
DEEP_GREEN = "#003c33"     # dark band
CORAL      = "#ff7759"     # accent
ACTION_BLUE = "#1863dc"    # info / links
MUTED      = "#93939f"     # secondary text
HAIRLINE   = "#d9d9dd"     # borders
PLOT_DPI   = 150

REGION_PALETTE = [
    DEEP_GREEN, CORAL, ACTION_BLUE, PRIMARY, "#6a994e", "#9b5de5", "#f4a261",
]

def apply_style():
    """Apply the project plot style."""
    try:
        plt.style.use("seaborn-v0_8-whitegrid")
    except OSError:
        plt.style.use("seaborn-whitegrid")
    return plt

apply_style()
print("🎨 Design system loaded!")

### 3.4 Distribusi Harga Petrol

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
values = df[TARGET].astype(float)
ax.hist(values, bins=40, color=DEEP_GREEN, alpha=0.85, edgecolor="white")
ax.axvline(values.mean(), color=CORAL, lw=2, ls="--",
           label=f"Mean = {values.mean():.2f}")
ax.axvline(values.median(), color=ACTION_BLUE, lw=2, ls=":",
           label=f"Median = {values.median():.2f}")
ax.set_title("Distribution of Petrol Price (USD/L)", color=PRIMARY,
             fontsize=13, fontweight="bold")
ax.set_xlabel("Petrol price (USD/L)", color=PRIMARY)
ax.set_ylabel("Frequency", color=PRIMARY)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

### 3.5 Time Series — Rata-rata Harga Bulanan

In [ ]:
tmp = df.copy()
tmp["date"] = pd.to_datetime(tmp["date"], errors="coerce")
tmp = tmp.dropna(subset=["date"])
monthly = (
    tmp.set_index("date")[TARGET]
    .resample("ME").mean()
    .dropna()
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly.index, monthly.values, color=DEEP_GREEN, lw=2)
ax.fill_between(monthly.index, monthly.values, color=DEEP_GREEN, alpha=0.12)
ax.set_title("Global Monthly-Average Petrol Price (2020–2026)",
             color=PRIMARY, fontsize=13, fontweight="bold")
ax.set_xlabel("Date", color=PRIMARY)
ax.set_ylabel("Avg petrol price (USD/L)", color=PRIMARY)
fig.tight_layout()
plt.show()

### 3.6 Distribusi Harga per Region

In [ ]:
regions = (
    df.groupby("region")[TARGET].median().sort_values(ascending=True).index
)
data = [df.loc[df["region"] == r, TARGET].astype(float).values for r in regions]

fig, ax = plt.subplots(figsize=(10, 6))
bp = ax.boxplot(data, vert=False, patch_artist=True, labels=list(regions),
                medianprops=dict(color=PRIMARY, linewidth=1.5),
                flierprops=dict(marker="o", markersize=3,
                                markerfacecolor=CORAL, alpha=0.4,
                                markeredgecolor="none"))
for patch, color in zip(bp["boxes"], REGION_PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
ax.set_title("Petrol Price Distribution by Region", color=PRIMARY,
             fontsize=13, fontweight="bold")
ax.set_xlabel("Petrol price (USD/L)", color=PRIMARY)
fig.tight_layout()
plt.show()

### 3.7 Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
corr = df.select_dtypes(include=[np.number]).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdYlGn",
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={"shrink": 0.8})
ax.set_title("Correlation Matrix (Numeric Features)", color=PRIMARY,
             fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

---
## 4. Preprocessing

### Feature Engineering

| Fitur | Metode Encoding |
|-------|----------------|
| `region` | One-Hot Encoding (`drop_first=True`) → 6 kolom |
| `income_level` | Ordinal (Low=0, Middle=1, High=2) |
| `subsidy_level` | Ordinal (Low=0, Medium=1, High=2, Very High=3) |
| `country` | Label Encoding (84 kategori, median fallback) |
| `brent_crude_usd` | StandardScaler |
| `tax_percentage` | StandardScaler |
| `year` | Diekstrak dari `date`, lalu StandardScaler |
| `month` | Diekstrak dari `date`, lalu StandardScaler |

**Target:** `petrol_usd_liter`

In [ ]:
class DataPreprocessor:
    """Encode raw fuel-price records into a model-ready numeric matrix."""

    TARGET = "petrol_usd_liter"
    INCOME_MAP = {"Low": 0, "Middle": 1, "High": 2}
    SUBSIDY_MAP = {"Low": 0, "Medium": 1, "High": 2, "Very High": 3}
    SCALED_COLS = ["brent_crude_usd", "tax_percentage", "year", "month"]

    def __init__(self):
        self.region_categories: List[str] = []
        self.region_columns: List[str] = []
        self.income_map = dict(self.INCOME_MAP)
        self.subsidy_map = dict(self.SUBSIDY_MAP)
        self.country_to_code: dict = {}
        self.country_median_code: int = 0
        self.scaler: StandardScaler = StandardScaler()
        self.feature_names: List[str] = []
        self.is_fitted: bool = False

    @staticmethod
    def _extract_date_parts(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        if "year" in df.columns and "month" in df.columns:
            return df
        if "date" not in df.columns:
            raise ValueError("Input frame must contain 'date' or 'year'+'month'.")
        parsed = pd.to_datetime(df["date"], errors="coerce")
        df["year"] = parsed.dt.year
        df["month"] = parsed.dt.month
        return df

    def _encode_region(self, df: pd.DataFrame) -> pd.DataFrame:
        out = pd.DataFrame(index=df.index)
        regions = df["region"].astype(str)
        for col in self.region_columns:
            category = col[len("region_"):]
            out[col] = (regions == category).astype(float)
        return out

    def _encode_country(self, series: pd.Series) -> np.ndarray:
        return series.astype(str).map(
            lambda c: self.country_to_code.get(c, self.country_median_code)
        ).astype(float).to_numpy()

    def _build_matrix(self, df: pd.DataFrame) -> np.ndarray:
        region_part = self._encode_region(df)
        income = df["income_level"].astype(str).map(self.income_map)
        subsidy = df["subsidy_level"].astype(str).map(self.subsidy_map)
        if income.isnull().any():
            raise ValueError(f"Unknown income_level: {df['income_level'][income.isnull()].unique()}")
        if subsidy.isnull().any():
            raise ValueError(f"Unknown subsidy_level: {df['subsidy_level'][subsidy.isnull()].unique()}")
        country = self._encode_country(df["country"])
        scaled = self.scaler.transform(df[self.SCALED_COLS].astype(float))
        return np.column_stack([
            region_part.to_numpy(),
            income.to_numpy(dtype=float),
            subsidy.to_numpy(dtype=float),
            country,
            scaled,
        ])

    def _compose_feature_names(self) -> List[str]:
        return (
            list(self.region_columns)
            + ["income_level", "subsidy_level", "country"]
            + list(self.SCALED_COLS)
        )

    def fit_transform(
        self, df: pd.DataFrame, test_size: float = 0.2,
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, List[str]]:
        """Fit pipeline and return train/test split."""
        logger.info("Fitting DataPreprocessor on %d records.", len(df))
        df = self._extract_date_parts(df)

        required = {
            "region", "income_level", "subsidy_level", "country",
            *self.SCALED_COLS, self.TARGET,
        }
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"Missing required columns: {sorted(missing)}")

        y_full = df[self.TARGET].astype(float)

        df_train, df_test, y_train, y_test = train_test_split(
            df, y_full, test_size=test_size,
            random_state=RANDOM_STATE, shuffle=True,
        )
        logger.info("Train/test split: %d train / %d test rows.",
                    len(df_train), len(df_test))

        # Learn vocabularies on training data only
        self.region_categories = sorted(df_train["region"].astype(str).unique())
        self.region_columns = [f"region_{c}" for c in self.region_categories[1:]]

        countries = sorted(df_train["country"].astype(str).unique())
        self.country_to_code = {c: i for i, c in enumerate(countries)}
        self.country_median_code = int(np.median(list(self.country_to_code.values())))

        self.scaler = StandardScaler()
        self.scaler.fit(df_train[self.SCALED_COLS].astype(float))

        self.feature_names = self._compose_feature_names()
        self.is_fitted = True

        X_train = self._build_matrix(df_train)
        X_test = self._build_matrix(df_test)
        y_train_arr = y_train.to_numpy(dtype=float)
        y_test_arr = y_test.to_numpy(dtype=float)

        logger.info("Design matrix shape: X_train=%s, X_test=%s",
                    X_train.shape, X_test.shape)

        return X_train, X_test, y_train_arr, y_test_arr, self.feature_names

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        if not self.is_fitted:
            raise RuntimeError("Preprocessor must be fitted first.")
        df = self._extract_date_parts(df)
        return self._build_matrix(df)

print("✅ DataPreprocessor class defined.")

In [ ]:
# Fit preprocessor dan buat train/test split
preprocessor = DataPreprocessor()
X_train, X_test, y_train, y_test, feature_names = preprocessor.fit_transform(df)

print(f"\n📐 Feature names ({len(feature_names)}):")
for i, name in enumerate(feature_names):
    print(f"   [{i:2d}] {name}")
print(f"\n   X_train shape: {X_train.shape}")
print(f"   X_test  shape: {X_test.shape}")
print(f"   y_train range: [{y_train.min():.3f}, {y_train.max():.3f}]")
print(f"   y_test  range: [{y_test.min():.3f}, {y_test.max():.3f}]")

---
## 5. Utility Functions — Metrics & Plots

In [ ]:
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    """Hitung MAE, MSE, RMSE, R², dan MAPE."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mse = float(mean_squared_error(y_true, y_pred))
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "MSE": mse,
        "RMSE": float(np.sqrt(mse)),
        "R2": float(r2_score(y_true, y_pred)),
        "MAPE": float(mean_absolute_percentage_error(y_true, y_pred) * 100.0),
    }


def plot_predictions_vs_actual(y_true, y_pred, model_name):
    """Scatter plot: Predicted vs Actual."""
    apply_style()
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    metrics = compute_metrics(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(y_true, y_pred, s=14, alpha=0.45, color=DEEP_GREEN, edgecolors="none")
    lo = float(min(y_true.min(), y_pred.min()))
    hi = float(max(y_true.max(), y_pred.max()))
    ax.plot([lo, hi], [lo, hi], color=CORAL, lw=2, ls="--", label="Ideal (y = x)")
    ax.set_title(f"{model_name} — Predicted vs Actual (R² = {metrics['R2']:.3f})",
                 color=PRIMARY, fontsize=13, fontweight="bold")
    ax.set_xlabel("Actual petrol price (USD/L)", color=PRIMARY)
    ax.set_ylabel("Predicted petrol price (USD/L)", color=PRIMARY)
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()


def plot_residuals(y_true, y_pred, model_name):
    """Residual plot."""
    apply_style()
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    residuals = y_true - y_pred

    fig, ax = plt.subplots(figsize=(8, 5.5))
    ax.scatter(y_pred, residuals, s=14, alpha=0.45, color=ACTION_BLUE, edgecolors="none")
    ax.axhline(0.0, color=CORAL, lw=2, ls="--")
    ax.set_title(f"{model_name} — Residual Plot", color=PRIMARY,
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("Predicted petrol price (USD/L)", color=PRIMARY)
    ax.set_ylabel("Residual (Actual − Predicted)", color=PRIMARY)
    fig.tight_layout()
    plt.show()


def plot_feature_importance(model, feature_names, top_n=15):
    """Horizontal bar chart of feature importances (Random Forest only)."""
    apply_style()
    importances = np.asarray(model.feature_importances_, dtype=float)
    names = np.asarray(feature_names)
    order = np.argsort(importances)[::-1][:top_n]
    top_names = names[order][::-1]
    top_values = importances[order][::-1]

    fig, ax = plt.subplots(figsize=(9, 7))
    bars = ax.barh(range(len(top_values)), top_values, color=DEEP_GREEN,
                   edgecolor=PRIMARY, linewidth=0.6)
    bars[-1].set_color(CORAL)
    ax.set_yticks(range(len(top_names)))
    ax.set_yticklabels(top_names, color=PRIMARY)
    ax.set_xlabel("Feature importance (impurity-based)", color=PRIMARY)
    ax.set_title(f"Random Forest — Top {len(top_names)} Feature Importances",
                 color=PRIMARY, fontsize=13, fontweight="bold")
    for i, v in enumerate(top_values):
        ax.text(v + max(top_values) * 0.01, i, f"{v:.3f}",
                va="center", color=PRIMARY, fontsize=8)
    fig.tight_layout()
    plt.show()


print("✅ Utility functions (compute_metrics, plots) defined.")

---
## 6. Model 1 — K-Nearest Neighbours (KNN)

KNN menggunakan `GridSearchCV` untuk mencari kombinasi terbaik dari:
- `n_neighbors`: [3, 4, 5, 7, 9, 11, 15, 20]
- `weights`: [uniform, distance]
- `metric`: [euclidean, manhattan]

Fitur `country` di-boost ×10 agar KNN lebih memprioritaskan neighbours dari negara yang sama.

In [ ]:
class _CountryBooster(BaseEstimator, TransformerMixin):
    """Multiply the label-encoded country column by a fixed weight."""
    def __init__(self, country_index: int, weight: float = 10.0):
        self.country_index = country_index
        self.weight = weight

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float).copy()
        X[:, self.country_index] *= self.weight
        return X


# ── KNN Training ──
print("=" * 60)
print("TRAINING KNN")
print("=" * 60)

KNN_PARAM_GRID = {
    "knn__n_neighbors": [3, 4, 5, 7, 9, 11, 15, 20],
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["euclidean", "manhattan"],
}

# Country boost
country_index = feature_names.index("country") if "country" in feature_names else None
knn_steps = []
if country_index is not None:
    knn_steps.append(("boost", _CountryBooster(country_index, weight=10.0)))
knn_steps.append(("knn", KNeighborsRegressor()))
knn_pipe = Pipeline(knn_steps)

start = time.perf_counter()
knn_search = GridSearchCV(
    estimator=knn_pipe,
    param_grid=KNN_PARAM_GRID,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
)
knn_search.fit(X_train, y_train)
knn_time = time.perf_counter() - start

knn_model = knn_search.best_estimator_
knn_best_params = {k.split("__", 1)[-1]: v for k, v in knn_search.best_params_.items()}

print(f"\n⏱️  Training time: {knn_time:.1f}s")
print(f"🔧 Best params: {knn_best_params}")
print(f"📊 Best CV RMSE: {np.sqrt(-knn_search.best_score_):.5f}")

In [ ]:
# ── KNN Evaluation ──
knn_pred = knn_model.predict(X_test)
knn_metrics = compute_metrics(y_test, knn_pred)

print("\n📈 KNN Test Metrics:")
for k, v in knn_metrics.items():
    print(f"   {k:>6s}: {v:.5f}")

plot_predictions_vs_actual(y_test, knn_pred, "KNN")
plot_residuals(y_test, knn_pred, "KNN")

---
## 7. Model 2 — Support Vector Regression (SVM)

SVM menggunakan strategi **tune small, refit big**:
1. Grid search 72 kombinasi pada subset kecil (1,500 rows)
2. Refit model terbaik pada subsample lebih besar (10,000 rows)

Kernel `linear` menggunakan `LinearSVR` yang jauh lebih cepat daripada `SVR(kernel='linear')`.

In [ ]:
# ── SVM Training ──
print("=" * 60)
print("TRAINING SVM")
print("=" * 60)

SVM_PARAM_GRID = {
    "kernel": ["rbf", "linear", "poly"],
    "C": [0.1, 1, 10, 100],
    "epsilon": [0.01, 0.1, 0.5],
    "gamma": ["scale", "auto"],
}
SVR_MAX_ITER = 5_000
LINEAR_MAX_ITER = 10_000

rng = np.random.RandomState(RANDOM_STATE)
n = len(X_train)
subsample_size = 10_000
tune_size = 1_500

# Final-fit subsample
if n > subsample_size:
    fit_idx = rng.choice(n, size=subsample_size, replace=False)
    X_fit, y_fit = X_train[fit_idx], y_train[fit_idx]
    print(f"   Final fit subsample: {subsample_size:,} of {n:,} rows")
else:
    X_fit, y_fit = X_train, y_train

# Tuning subset
m = len(X_fit)
if m > tune_size:
    tune_idx = rng.choice(m, size=tune_size, replace=False)
    X_tune, y_tune = X_fit[tune_idx], y_fit[tune_idx]
else:
    X_tune, y_tune = X_fit, y_fit

cs = SVM_PARAM_GRID["C"]
eps = SVM_PARAM_GRID["epsilon"]
gammas = SVM_PARAM_GRID["gamma"]

start = time.perf_counter()
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)

    # Search A: SVR — rbf + poly
    print(f"   Tuning rbf/poly on {len(X_tune)} rows (cv=5)...")
    svr_grid = {"kernel": ["rbf", "poly"], "C": cs, "epsilon": eps, "gamma": gammas}
    svr_search = GridSearchCV(
        SVR(cache_size=1000, max_iter=SVR_MAX_ITER), svr_grid,
        cv=5, scoring="neg_mean_squared_error", n_jobs=-1, refit=False)
    svr_search.fit(X_tune, y_tune)
    print(f"   Best rbf/poly: {svr_search.best_params_} "
          f"(RMSE={np.sqrt(-svr_search.best_score_):.5f})")

    # Search B: LinearSVR
    print(f"   Tuning linear on {len(X_tune)} rows (cv=5)...")
    lin_grid = {"C": cs, "epsilon": eps}
    lin_search = GridSearchCV(
        LinearSVR(max_iter=LINEAR_MAX_ITER, random_state=RANDOM_STATE),
        lin_grid, cv=5, scoring="neg_mean_squared_error",
        n_jobs=-1, refit=False)
    lin_search.fit(X_tune, y_tune)
    print(f"   Best linear: {lin_search.best_params_} "
          f"(RMSE={np.sqrt(-lin_search.best_score_):.5f})")

    # Pick the better search
    if svr_search.best_score_ >= lin_search.best_score_:
        svm_params = dict(svr_search.best_params_)
        svm_best_score = float(svr_search.best_score_)
        svm_model = SVR(cache_size=1000, max_iter=SVR_MAX_ITER, **svm_params)
        svm_best_params = svm_params
    else:
        svm_params = dict(lin_search.best_params_)
        svm_best_score = float(lin_search.best_score_)
        svm_model = LinearSVR(max_iter=LINEAR_MAX_ITER,
                              random_state=RANDOM_STATE, **svm_params)
        svm_best_params = {"kernel": "linear", **svm_params}

    # Refit on full subsample
    print(f"   Refitting best model on {len(X_fit)} rows...")
    svm_model.fit(X_fit, y_fit)

svm_time = time.perf_counter() - start

print(f"\n⏱️  Training time: {svm_time:.1f}s")
print(f"🔧 Best params: {svm_best_params}")
print(f"📊 Best CV RMSE: {np.sqrt(-svm_best_score):.5f}")

In [ ]:
# ── SVM Evaluation ──
svm_pred = svm_model.predict(X_test)
svm_metrics = compute_metrics(y_test, svm_pred)

print("\n📈 SVM Test Metrics:")
for k, v in svm_metrics.items():
    print(f"   {k:>6s}: {v:.5f}")

plot_predictions_vs_actual(y_test, svm_pred, "SVM")
plot_residuals(y_test, svm_pred, "SVM")

---
## 8. Model 3 — Random Forest

Random Forest menggunakan `RandomizedSearchCV` (30 iterasi) untuk mencari parameter terbaik dari distribusi:
- `n_estimators`: [50, 100, 200, 300]
- `max_depth`: [None, 10, 20, 30, 50]
- `min_samples_split`: [2, 5, 10]
- `min_samples_leaf`: [1, 2, 4]
- `max_features`: [sqrt, log2, None]
- `bootstrap`: [True, False]

In [ ]:
# ── Random Forest Training ──
print("=" * 60)
print("TRAINING RANDOM FOREST")
print("=" * 60)

RF_PARAM_DIST = {
    "n_estimators": [50, 100, 200, 300],
    "max_depth": [None, 10, 20, 30, 50],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None],
    "bootstrap": [True, False],
}

start = time.perf_counter()
rf_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=RANDOM_STATE),
    param_distributions=RF_PARAM_DIST,
    n_iter=30,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
rf_search.fit(X_train, y_train)
rf_time = time.perf_counter() - start

rf_model = rf_search.best_estimator_
rf_best_params = rf_search.best_params_

print(f"\n⏱️  Training time: {rf_time:.1f}s")
print(f"🔧 Best params: {rf_best_params}")
print(f"📊 Best CV RMSE: {np.sqrt(-rf_search.best_score_):.5f}")

In [ ]:
# ── Random Forest Evaluation ──
rf_pred = rf_model.predict(X_test)
rf_metrics = compute_metrics(y_test, rf_pred)

print("\n📈 Random Forest Test Metrics:")
for k, v in rf_metrics.items():
    print(f"   {k:>6s}: {v:.5f}")

plot_predictions_vs_actual(y_test, rf_pred, "Random Forest")
plot_residuals(y_test, rf_pred, "Random Forest")
plot_feature_importance(rf_model, feature_names)

---
## 9. Perbandingan Model

In [ ]:
# ── Build comparison table ──
comparison = {
    "KNN": {"metrics": knn_metrics, "best_params": knn_best_params,
            "train_seconds": round(knn_time, 2)},
    "SVM": {"metrics": svm_metrics, "best_params": svm_best_params,
            "train_seconds": round(svm_time, 2)},
    "Random Forest": {"metrics": rf_metrics, "best_params": rf_best_params,
                      "train_seconds": round(rf_time, 2)},
}

# Determine best model
best_name = min(comparison, key=lambda m: comparison[m]["metrics"]["RMSE"])
best_rmse = comparison[best_name]["metrics"]["RMSE"]

# Print summary table
header = f"{'Model':<16}{'MAE':>10}{'RMSE':>10}{'R²':>10}{'MAPE %':>12}{'Akurasi%':>11}"
line = "-" * len(header)
print("\n" + line)
print("MODEL COMPARISON SUMMARY".center(len(header)))
print(line)
print(header)
print(line)
for m, data in comparison.items():
    mt = data["metrics"]
    print(f"{m:<16}{mt['MAE']:>10.4f}{mt['RMSE']:>10.4f}"
          f"{mt['R2']:>10.4f}{mt['MAPE']:>12.2f}{mt['R2'] * 100:>11.2f}")
print(line)
print(f"🏆 BEST MODEL (lowest RMSE): {best_name} (RMSE = {best_rmse:.4f})")
print(line)

In [ ]:
# ── Comparison Bar Charts ──
apply_style()
models = list(comparison.keys())
colours = [DEEP_GREEN, CORAL, ACTION_BLUE]

panels = [
    ("MAE", "MAE (USD/L) — lower is better", False),
    ("RMSE", "RMSE (USD/L) — lower is better", False),
    ("R2", "R² — higher is better", True),
    ("MAPE", "MAPE (%) — lower is better", False),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle("Model Performance Comparison", fontsize=16,
             fontweight="bold", color=PRIMARY)

for ax, (metric, title, higher_better) in zip(axes.ravel(), panels):
    values = [comparison[m]["metrics"][metric] for m in models]
    bars = ax.bar(models, values, color=colours, edgecolor=PRIMARY,
                  linewidth=0.8, width=0.6)
    best_idx = int(np.argmax(values)) if higher_better else int(np.argmin(values))
    bars[best_idx].set_edgecolor(CORAL)
    bars[best_idx].set_linewidth(2.5)
    ax.set_title(title, color=PRIMARY, fontsize=12, fontweight="bold")
    ax.set_ylabel(metric, color=PRIMARY)
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height(), f"{v:.3f}", ha="center", va="bottom",
                fontsize=9, color=PRIMARY)
    ax.margins(y=0.15)

fig.tight_layout(rect=(0, 0, 1, 0.97))
plt.show()

In [ ]:
# ── Training Time Comparison ──
apply_style()
times = [comparison[m]["train_seconds"] for m in models]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(models, times, color=colours, edgecolor=PRIMARY, linewidth=0.8, width=0.5)
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height(), f"{t:.1f}s", ha="center", va="bottom",
            fontsize=10, color=PRIMARY, fontweight="bold")
ax.set_title("Training Time Comparison", color=PRIMARY, fontsize=13, fontweight="bold")
ax.set_ylabel("Time (seconds)", color=PRIMARY)
ax.margins(y=0.15)
fig.tight_layout()
plt.show()

In [ ]:
# ── Comparison DataFrame ──
comp_df = pd.DataFrame({
    "Model": models,
    "MAE": [comparison[m]["metrics"]["MAE"] for m in models],
    "RMSE": [comparison[m]["metrics"]["RMSE"] for m in models],
    "R²": [comparison[m]["metrics"]["R2"] for m in models],
    "MAPE (%)": [comparison[m]["metrics"]["MAPE"] for m in models],
    "Akurasi (R²×100)": [comparison[m]["metrics"]["R2"] * 100 for m in models],
    "Training Time (s)": [comparison[m]["train_seconds"] for m in models],
}).round(5)

comp_df.style.highlight_min(
    subset=["MAE", "RMSE", "MAPE (%)", "Training Time (s)"],
    color="#c6efce"
).highlight_max(
    subset=["R²", "Akurasi (R²×100)"],
    color="#c6efce"
)

---
## 10. Kesimpulan

### Ringkasan Hasil

In [ ]:
print("=" * 60)
print("KESIMPULAN")
print("=" * 60)
print(f"""
Dataset : {df.shape[0]:,} records, {df['country'].nunique()} negara, {df['region'].nunique()} region
Periode : {df['date'].min()} s/d {df['date'].max()}
Target  : petrol_usd_liter
Split   : 80% train / 20% test (random_state={RANDOM_STATE})
Features: {len(feature_names)} fitur setelah encoding

🏆 Model Terbaik: {best_name}
   - RMSE  : {comparison[best_name]['metrics']['RMSE']:.5f} USD/L
   - R²    : {comparison[best_name]['metrics']['R2']:.5f}
   - MAE   : {comparison[best_name]['metrics']['MAE']:.5f} USD/L
   - MAPE  : {comparison[best_name]['metrics']['MAPE']:.2f}%
   - Akurasi: {comparison[best_name]['metrics']['R2'] * 100:.2f}%

Semua model mencapai R² > 0.99, menunjukkan bahwa fitur-fitur
yang tersedia (country, region, income level, subsidy level,
Brent crude price, tax percentage) sangat efektif untuk
memprediksi harga petrol per liter.
""")
print("=" * 60)

In [ ]:
# ── Save comparison result as JSON ──
result = {
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "models": {
        name: {
            "metrics": {k: round(v, 5) for k, v in data["metrics"].items()},
            "best_params": {str(k): (str(v) if v is not None else None)
                           for k, v in data["best_params"].items()},
            "train_seconds": data["train_seconds"],
        }
        for name, data in comparison.items()
    },
    "best_model": {
        "name": best_name,
        "by": "RMSE",
        "RMSE": round(best_rmse, 5),
    },
    "feature_names": feature_names,
}

with open("model_comparison.json", "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2)

print("💾 Saved: model_comparison.json")
print(json.dumps(result, indent=2))